# RHNA & Housing Production

RHNA targets and housing production (permits/completions) for the 18
incorporated jurisdictions in San Diego County plus the County itself,
pulled from HCD, DOF, City of San Diego, and Census sources.

Target year: 2025 for RHNA/APR/DOF/permits. ACS uses the 2020-2024
5-year vintage (its most recent release).


## Setup

In [66]:
%pip install pandas numpy requests openpyxl

Note: you may need to restart the kernel to use updated packages.


In [67]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import requests
from IPython.display import display


def find_workstream_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
        my_folder = candidate / "van's work"
        if (my_folder / "data").exists() and (my_folder / "notebooks").exists():
            return my_folder
    raise FileNotFoundError("Could not locate workstream root.")


ROOT = find_workstream_root()
RAW_DIR = ROOT / "data" / "raw" / "hcd"
PROCESSED_DIR = ROOT / "data" / "processed"
DOCS_DIR = ROOT / "docs"

for d in (RAW_DIR, PROCESSED_DIR, DOCS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Workstream root:", ROOT)


Workstream root: /Users/ice/Documents/GitHub/chpd-dashboard-data-validation/van's work


In [68]:
TARGET_YEAR = 2025
ACS_VINTAGE_LABEL = "2020-2024"
ACS_DATA_YEAR = 2024

SAN_DIEGO_CITIES = [
    "Carlsbad", "Chula Vista", "Coronado", "Del Mar", "El Cajon",
    "Encinitas", "Escondido", "Imperial Beach", "La Mesa", "Lemon Grove",
    "National City", "Oceanside", "Poway", "San Diego", "San Marcos",
    "Santee", "Solana Beach", "Vista",
]
COUNTY_JURISDICTION_NAME = "San Diego County"


In [69]:
def normalize_jurisdiction(name: object) -> str:
    s = str(name).strip().lower()
    if s == "national city":
        return "national city"
    s = re.sub(r"^city\s+of\s+", "", s)
    s = re.sub(r"^county\s+of\s+", "county ", s)
    s = re.sub(r"\s+city$", "", s)
    s = re.sub(r"[^a-z0-9]+", " ", s).strip()
    return s


sd_jur_keys = {normalize_jurisdiction(c) for c in SAN_DIEGO_CITIES}
sd_jur_keys.add(normalize_jurisdiction(COUNTY_JURISDICTION_NAME))


def get_json(url: str, params: dict | None = None, timeout: int = 60):
    response = requests.get(
        url, params=params, timeout=timeout,
        headers={"User-Agent": "CHPD Housing Dashboard Data Validation"},
    )
    if not response.ok:
        raise RuntimeError(f"Request failed ({response.status_code}): {response.url}")
    return response.json()


def find_resource_download_url(package_json: dict, name_contains: str):
    resources = package_json["result"]["resources"]
    matches = [
        r for r in resources
        if name_contains.lower() in r.get("name", "").lower()
        and r.get("format", "").upper() == "CSV"
    ]
    if not matches:
        raise ValueError(f"No CSV resource matching '{name_contains}' found.")
    matches.sort(key=lambda r: r.get("last_modified", ""), reverse=True)
    return matches[0]["url"], matches[0]["name"]


## APR (permits, entitlements, completions)

In [70]:
CKAN_PACKAGE_URL = (
    "https://data.ca.gov/api/3/action/package_show"
    "?id=housing-element-annual-progress-report-apr-data-by-jurisdiction-and-year"
)
package_json = get_json(CKAN_PACKAGE_URL)
table_a2_url, table_a2_name = find_resource_download_url(package_json, "Table A2")

raw_path = RAW_DIR / "apr_table_a2_raw.csv"
if not raw_path.exists():
    resp = requests.get(table_a2_url, timeout=300)
    resp.raise_for_status()
    raw_path.write_bytes(resp.content)

apr_raw = pd.read_csv(raw_path, low_memory=False)
print(apr_raw.shape)
apr_raw.head()


(920915, 69)


,JURIS_NAME,CNTY_NAME,YEAR,PRIOR_APN,APN,STREET_ADDRESS,PROJECT_NAME,JURS_TRACKING_ID,UNIT_CAT,TENURE,...,DEM_DES_UNITS_OWN_RENT,DENSITY_BONUS_TOTAL,DENSITY_BONUS_NUMBER_OTHER_INCENTIVES,DENSITY_BONUS_INCENTIVES,DENSITY_BONUS_RECEIVE_REDUCTION,NOTES,LATITUDE,LONGITUDE,STD_ADDRESS,SCORE
0,STANISLAUS COUNTY,Stanislaus,2020,NaN,081-002-032,1464 CLARK RD,NaN,BLD2002-01052,MH,Renter,...,0,0.0,0,NaN,NaN,MOBILE HOME ON PRIVATE PROPERTY (PIERS) 2002 G...,37.655693,-121.085881,"1464 Clark Rd, Modesto, California, 95358",93.96
1,STANISLAUS COUNTY,Stanislaus,2020,NaN,002-024-046,15363 ORANGE BLOSSOM RD,NaN,BLD2020-0101,MH,Renter,...,0,0.0,0,NaN,NaN,TEMPORARY 1624 SQ FT MANUFACTURED DWELLING - 2...,37.815710,-120.715241,"15363 Orange Blossom Rd, Oakdale, California, ...",89.68
2,STANISLAUS COUNTY,Stanislaus,2020,NaN,048-006-007,2333 FIG AVE,NaN,BLD2020-1248,MH,Renter,...,0,0.0,0,NaN,NaN,(( ACA )) MANUFACTURED HOME 2020 CHAMPION MODE...,37.477489,-121.080866,"2333 Fig Ave, Patterson, California, 95363",87.69
3,STANISLAUS COUNTY,Stanislaus,2020,NaN,001-009-015,6990 State Route 4,NaN,BLD2018-1774,MH,Owner,...,0,0.0,0,NaN,NaN,MANUFACTURED HOME / 2018 CMH MODEL FAIRPOINT 2...,37.490499,-120.848087,"4th St, Turlock, California, 95380",80.09
4,STANISLAUS COUNTY,Stanislaus,2020,NaN,062-027-003,5314 LANGWORTH,NaN,BLD2019-1749,MH,Renter,...,0,0.0,0,NaN,NaN,TEMPORARY 1440 SQ FT MANUFACTED DWELLING 1973 ...,37.717510,-120.894040,"5314 Langworth Rd, Oakdale, California, 95361",87.17


In [71]:
print(apr_raw.columns.tolist())


['JURIS_NAME', 'CNTY_NAME', 'YEAR', 'PRIOR_APN', 'APN', 'STREET_ADDRESS', 'PROJECT_NAME', 'JURS_TRACKING_ID', 'UNIT_CAT', 'TENURE', 'ACUTELY_LOW_INCOME_DR', 'ACUTELY_LOW_INCOME_NDR', 'EXTREMELY_LOW_INCOME_DR', 'EXTREMELY_LOW_INCOME_NDR', 'VLOW_INCOME_DR', 'VLOW_INCOME_NDR', 'LOW_INCOME_DR', 'LOW_INCOME_NDR', 'MOD_INCOME_DR', 'MOD_INCOME_NDR', 'ABOVE_MOD_INCOME', 'ENT_APPROVE_DT1', 'NO_ENTITLEMENTS', 'BP_ACUTELY_LOW_INCOME_DR', 'BP_ACUTELY_LOW_INCOME_NDR', 'BP_EXTREMELY_LOW_INCOME_DR', 'BP_EXTREMELY_LOW_INCOME_NDR', 'BP_VLOW_INCOME_DR', 'BP_VLOW_INCOME_NDR', 'BP_LOW_INCOME_DR', 'BP_LOW_INCOME_NDR', 'BP_MOD_INCOME_DR', 'BP_MOD_INCOME_NDR', 'BP_ABOVE_MOD_INCOME', 'BP_ISSUE_DT1', 'NO_BUILDING_PERMITS', 'CO_ACUTELY_LOW_INCOME_DR', 'CO_ACUTELY_LOW_INCOME_NDR', 'CO_EXTREMELY_LOW_INCOME_DR', 'CO_EXTREMELY_LOW_INCOME_NDR', 'CO_VLOW_INCOME_DR', 'CO_VLOW_INCOME_NDR', 'CO_LOW_INCOME_DR', 'CO_LOW_INCOME_NDR', 'CO_MOD_INCOME_DR', 'CO_MOD_INCOME_NDR', 'CO_ABOVE_MOD_INCOME', 'CO_ISSUE_DT1', 'NO_OTHER_

In [72]:
JURISDICTION_COL = "JURIS_NAME"
YEAR_COL = "YEAR"

apr_raw["jur_clean"] = apr_raw[JURISDICTION_COL].map(normalize_jurisdiction)
apr_raw[YEAR_COL] = pd.to_numeric(apr_raw[YEAR_COL], errors="coerce")

sd_apr = apr_raw[apr_raw["jur_clean"].isin(sd_jur_keys)].copy()
sd_apr_target_year = sd_apr[sd_apr[YEAR_COL] == TARGET_YEAR].copy()

print("SD rows, all years:", len(sd_apr))
print(f"SD rows, {TARGET_YEAR}:", len(sd_apr_target_year))
missing = sd_jur_keys - set(sd_apr_target_year["jur_clean"].unique())
if missing:
    print(f"No {TARGET_YEAR} row yet:", sorted(missing))


SD rows, all years: 52514
SD rows, 2025: 7460


In [73]:
BP_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("BP_") and "INCOME" in c.upper()]
CO_INCOME_COLS = [c for c in sd_apr_target_year.columns if c.upper().startswith("CO_") and "INCOME" in c.upper()]

sd_apr_target_year["bp_units_row"] = sd_apr_target_year[BP_INCOME_COLS].sum(axis=1, numeric_only=True)
sd_apr_target_year["co_units_row"] = sd_apr_target_year[CO_INCOME_COLS].sum(axis=1, numeric_only=True)

above_mod_bp = [c for c in BP_INCOME_COLS if "ABOVE" in c.upper()]
above_mod_co = [c for c in CO_INCOME_COLS if "ABOVE" in c.upper()]

sd_apr_target_year["bp_affordable_row"] = (
    sd_apr_target_year["bp_units_row"] - sd_apr_target_year[above_mod_bp].sum(axis=1, numeric_only=True)
)
sd_apr_target_year["co_affordable_row"] = (
    sd_apr_target_year["co_units_row"] - sd_apr_target_year[above_mod_co].sum(axis=1, numeric_only=True)
)

# The raw file is row-level (one row per project/address) -- group up to
# jurisdiction-year before this becomes a usable production metric.
production_by_year = (
    sd_apr_target_year
    .groupby("jur_clean", as_index=False)
    .agg(
        year=(YEAR_COL, "first"),
        bp_units_total=("bp_units_row", "sum"),
        co_units_total=("co_units_row", "sum"),
        bp_affordable_total=("bp_affordable_row", "sum"),
        co_affordable_total=("co_affordable_row", "sum"),
        project_rows=("jur_clean", "size"),
    )
)
production_by_year["bp_affordable_share"] = production_by_year["bp_affordable_total"] / production_by_year["bp_units_total"]
production_by_year["co_affordable_share"] = production_by_year["co_affordable_total"] / production_by_year["co_units_total"]
production_by_year


,jur_clean,year,bp_units_total,co_units_total,bp_affordable_total,co_affordable_total,project_rows,bp_affordable_share,co_affordable_share
0,carlsbad,2025,343,719,36,141,428,0.104956,0.196106
1,chula vista,2025,723,1428,224,201,663,0.309820,0.140756
2,coronado,2025,24,36,0,0,54,0.000000,0.000000
3,del mar,2025,14,17,11,10,43,0.785714,0.588235
4,el cajon,2025,210,235,83,123,164,0.395238,0.523404
5,encinitas,2025,187,228,33,43,461,0.176471,0.188596
6,escondido,2025,514,604,246,66,396,0.478599,0.109272
7,imperial beach,2025,71,13,0,0,86,0.000000,0.000000
8,la mesa,2025,254,249,99,215,223,0.389764,0.863454
9,lemon grove,2025,32,61,6,8,77,0.187500,0.131148


## RHNA 6th Cycle targets

In [74]:
RHNA_PACKAGE_URL = "https://data.ca.gov/api/3/action/package_show?id=rhna-progress-report"
rhna_package_json = get_json(RHNA_PACKAGE_URL)
rhna6_url, rhna6_name = find_resource_download_url(rhna_package_json, "6th Cycle RHNA Progress Report")

rhna_raw_path = RAW_DIR / "rhna6_progress_raw.csv"
if not rhna_raw_path.exists():
    resp = requests.get(rhna6_url, timeout=120)
    resp.raise_for_status()
    rhna_raw_path.write_bytes(resp.content)

rhna_raw = pd.read_csv(rhna_raw_path, low_memory=False)
print(rhna_raw.shape)
rhna_raw.head()


(539, 15)


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %
0,AMERICAN CANYON,01/31/2023 - 01/31/2031,True,11,169,0.07,5,109,0.05,2,95,0.02,487,249,1.96
1,AGOURA HILLS,10/15/2021 - 10/15/2029,True,44,127,0.35,10,72,0.14,6,55,0.11,277,64,4.33
2,ALPINE COUNTY,08/31/2019 - 06/30/2024,True,0,1,0.00,0,1,0.00,2,0,0.00,33,0,0.00
3,ALAMEDA,01/31/2023 - 01/31/2031,True,155,1421,0.11,47,818,0.06,55,868,0.06,192,2246,0.09
4,AMADOR,09/15/2021 - 09/15/2029,True,0,1,0.00,0,1,0.00,0,1,0.00,3,2,1.50


In [75]:
print(rhna_raw.columns.tolist())

['Jurisdiction', 'Planning Period', '6th Cycle Started', 'VLI UNITS', 'RHNA VLI', 'VLI %', 'LI UNITS', 'RHNA LI', 'LI %', 'MOD UNITS', 'RHNA MOD', 'MOD %', 'ABOVE MOD UNITS', 'RHNA ABOVE MOD', 'ABOVE MOD %']


In [76]:
RHNA_JUR_COL = "Jurisdiction"  # confirm against columns above

rhna_raw["jur_clean"] = rhna_raw[RHNA_JUR_COL].map(normalize_jurisdiction)
sd_rhna6 = rhna_raw[rhna_raw["jur_clean"].isin(sd_jur_keys)].copy()

print("SD rows:", len(sd_rhna6))
missing = sd_jur_keys - set(sd_rhna6["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_rhna6.head()


SD rows: 19


,Jurisdiction,Planning Period,6th Cycle Started,VLI UNITS,RHNA VLI,VLI %,LI UNITS,RHNA LI,LI %,MOD UNITS,RHNA MOD,MOD %,ABOVE MOD UNITS,RHNA ABOVE MOD,ABOVE MOD %,jur_clean
76,CARLSBAD,04/30/2021 - 04/30/2029,True,65,1311,0.05,198,784,0.25,293,749,0.39,991,1029,0.96,carlsbad
85,CORONADO,04/30/2021 - 04/30/2029,True,0,312,0.00,0,169,0.00,0,159,0.00,204,272,0.75,coronado
92,DEL MAR,04/30/2021 - 04/30/2029,True,0,37,0.00,0,64,0.00,66,31,2.13,45,31,1.45,del mar
99,EL CAJON,04/30/2021 - 04/30/2029,True,0,481,0.00,281,414,0.68,154,518,0.30,350,1867,0.19,el cajon
151,CHULA VISTA,04/30/2021 - 04/30/2029,True,130,2750,0.05,377,1777,0.21,943,1911,0.49,4716,4667,1.01,chula vista


## City of San Diego permits

In [77]:
permits_dir = RAW_DIR.parent / "sandiego_permits"
permits_dir.mkdir(parents=True, exist_ok=True)

permits_frames = []
for label in ["active", "closed"]:
    raw_path = permits_dir / f"{label}_approvals_raw.csv"
    if not raw_path.exists():
        print(f"Missing: {raw_path}")
        print("Download from https://data.sandiego.gov/datasets/development-permits-set2/ and save it there.")
        continue
    df = pd.read_csv(raw_path, low_memory=False)
    df["approval_status"] = label
    permits_frames.append(df)

if permits_frames:
    sd_permits_raw = pd.concat(permits_frames, ignore_index=True)
    print(sd_permits_raw.shape)
else:
    sd_permits_raw = pd.DataFrame()
sd_permits_raw.head()


(1245947, 55)


,DEVELOPMENT_ID,PROJECT_ID,PROJECT_TYPE,PROJECT_STATUS,PROJECT_PROCESSING_CODE,PROJECT_CREATE_DATE,PROJECT_DEEMEDCOMPLETE_DATE,PROJECT_TRUST_ACCOUNT_NO,PROJECT_TITLE,PROJECT_SCOPE,...,APPROVAL_ADU_TOTAL,APPROVAL_JADU_EXTREMELY_LOW,APPROVAL_JADU_VERY_LOW,APPROVAL_JADU_LOW,APPROVAL_JADU_MODERATE,APPROVAL_JADU_ABOVE_MODERATE,APPROVAL_JADU_BONUS,APPROVAL_JADU_TOTAL,APPROVAL_PERMIT_HOLDER,approval_status
0,69849.0,84160,NaN,Closed,Standard,2005-09-16,NaN,NaN,TCP 43063,GASLAMP STREET REHABILITATION PROJECT PHASE 1 ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
1,66412.0,79510,NaN,Inspecting,Standard,2005-07-27,2005-07-27,NaN,Eddie Bauer T.I.Permit,5984 sq ft tenant improvement for Eddie Bauer ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Arrow Automatic Fire Sprinkler, Arrow Automat...",active
2,69631.0,83867,NaN,Closed,Standard,2005-09-14,2005-09-14,NaN,Robinson Tenant Improvement,SOUTHEASTERN SAN DIEGO.Building Permit. Repair...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
3,67560.0,83802,NaN,Inspecting,Standard,2005-09-13,2005-09-13,NaN,Metro C & O - 33rd St Gate,SESDPD; - I-1(Customer provided PIC by PGB) ;R...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active
4,41510.0,84276,NaN,In Review,Standard,2005-09-19,2005-09-23,NaN,4001 Illinois St Fourplex,GREATER NORTH PARK. Building Permit for new ...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,active


In [78]:
print(sd_permits_raw.columns.tolist())

['DEVELOPMENT_ID', 'PROJECT_ID', 'PROJECT_TYPE', 'PROJECT_STATUS', 'PROJECT_PROCESSING_CODE', 'PROJECT_CREATE_DATE', 'PROJECT_DEEMEDCOMPLETE_DATE', 'PROJECT_TRUST_ACCOUNT_NO', 'PROJECT_TITLE', 'PROJECT_SCOPE', 'JOB_ID', 'JOB_DRAWING_NUMBER', 'GIS_ADDRESS', 'GIS_APN', 'JOB_BC_CODE', 'JOB_BC_CODE_DESCRIPTION', 'GIS_LATITUDE', 'GIS_LONGITUDE', 'APPROVAL_ID', 'APPROVAL_CATEGORY_CODE', 'APPROVAL_PROCESSING_CODE', 'APPROVAL_TYPE', 'APPROVAL_STATUS', 'APPROVAL_SCOPE', 'APPROVAL_CREATE_DATE', 'APPROVAL_ISSUE_DATE', 'APPROVAL_CLOSE_DATE', 'APPROVAL_EXPIRE_DATE', 'APPROVAL_VALUATION', 'APPROVAL_DU_NET_CHANGE', 'APPROVAL_STORIES', 'APPROVAL_FLOOR_AREA', 'APPROVAL_DU_EXTREMELY_LOW', 'APPROVAL_DU_VERY_LOW', 'APPROVAL_DU_LOW', 'APPROVAL_DU_MODERATE', 'APPROVAL_DU_ABOVE_MODERATE', 'APPROVAL_DU_FUTURE_DEMO', 'APPROVAL_DU_BONUS', 'APPROVAL_ADU_EXTREMELY_LOW', 'APPROVAL_ADU_VERY_LOW', 'APPROVAL_ADU_LOW', 'APPROVAL_ADU_MODERATE', 'APPROVAL_ADU_ABOVE_MODERATE', 'APPROVAL_ADU_BONUS', 'APPROVAL_ADU_TOTAL', 

In [79]:
# The raw file mixes every permit type the City issues (electrical, plumbing,
# traffic, signage, etc.) with actual housing construction. Filter to rows
# that report a nonzero dwelling-unit impact, rather than text-matching
# APPROVAL_TYPE / JOB_BC_CODE_DESCRIPTION, since those categories are
# inconsistent (e.g. "Add/Alt Companion Unit/Acc Apt" adds a unit despite
# the "No Chg DU" naming pattern used elsewhere).
sd_permits_raw["APPROVAL_ISSUE_DATE"] = pd.to_datetime(sd_permits_raw["APPROVAL_ISSUE_DATE"], errors="coerce")

du_cols = ["APPROVAL_DU_NET_CHANGE", "APPROVAL_ADU_TOTAL", "APPROVAL_JADU_TOTAL"]
has_du_impact = sd_permits_raw[du_cols].fillna(0).ne(0).any(axis=1)
in_target_year = sd_permits_raw["APPROVAL_ISSUE_DATE"].dt.year == TARGET_YEAR

sd_permits_housing = sd_permits_raw[has_du_impact & in_target_year].copy()
print(f"Housing-relevant permits, {TARGET_YEAR}:", len(sd_permits_housing))
print("Of", len(sd_permits_raw), "total raw rows")
sd_permits_housing[["PROJECT_TITLE", "JOB_BC_CODE_DESCRIPTION", "APPROVAL_DU_NET_CHANGE", "APPROVAL_ADU_TOTAL", "APPROVAL_JADU_TOTAL"]].head(10)


Housing-relevant permits, 2025: 1512
Of 1245947 total raw rows


,PROJECT_TITLE,JOB_BC_CODE_DESCRIPTION,APPROVAL_DU_NET_CHANGE,APPROVAL_ADU_TOTAL,APPROVAL_JADU_TOTAL
262976,General-Express-Building Construction:6757/Roe...,Five or More Family Apt,NaN,9.0,0.0
263074,General-Standard-Building Construction:1874/Su...,Add/Alt Companion Unit/Acc Apt,NaN,1.0,0.0
263207,General-Standard-Combination Building Permit:2...,Add/Alt Companion Unit/Acc Apt,NaN,2.0,0.0
263424,General-Standard-Building Construction:4248/Arden,Add/Alt Companion Unit/Acc Apt,NaN,1.0,0.0
263480,General-Standard-Building Construction:1405/Grove,Add/Alt Companion Unit/Acc Apt,NaN,1.0,0.0
263720,General-Standard-Building Construction:3321/52nd,Add/Alt Companion Unit/Acc Apt,NaN,1.0,0.0
263756,General-Standard-Building Construction:6520/Be...,Add/Alt Companion Unit/Acc Apt,NaN,1.0,0.0
263948,General-Express-Building Construction:4088/Cro...,"Add/Alt 3+ Fam, Increase DU",NaN,10.0,0.0
264177,General-Standard-Building Construction:2155/Ba...,Add/Alt Companion Unit/Acc Apt,NaN,2.0,0.0
264501,General-Standard-Building Construction:3305/Mc...,One Family Detached,NaN,2.0,0.0


## CA DOF population & housing estimates (E-5)

In [80]:
DOF_RAW_PATH = RAW_DIR.parent / "dof" / "e5_population_housing.xlsx"
DOF_SHEET_NAME = f"E5CityCounty{TARGET_YEAR}"

dof_raw = pd.read_excel(DOF_RAW_PATH, sheet_name=DOF_SHEET_NAME, header=3)
dof_raw = dof_raw.rename(columns={"County/City/State": "name"})
dof_raw["name"] = dof_raw["name"].astype(str).str.strip()

dof_raw["is_county_header"] = dof_raw["Total"].isna() & dof_raw["name"].str.contains("County", na=False)
dof_raw["county"] = dof_raw["name"].where(dof_raw["is_county_header"]).ffill()

sd_dof = dof_raw[
    (dof_raw["county"] == "San Diego County")
    & (~dof_raw["is_county_header"])
    & (dof_raw["Total"].notna())
    & (~dof_raw["name"].isin(["Incorporated", "County Total"]))
].copy()

sd_dof["jur_clean"] = sd_dof["name"].map(
    lambda n: "county san diego" if n.strip() == "Unincorporated" else normalize_jurisdiction(n)
)
sd_dof["year"] = TARGET_YEAR

print("SD rows:", len(sd_dof))
missing = sd_jur_keys - set(sd_dof["jur_clean"].unique())
if missing:
    print("Not found:", sorted(missing))
sd_dof[["name", "Total", "Household", "Group Quarters"]]


SD rows: 19
Not found: ['san diego county']


,name,Total,Household,Group Quarters
615,Carlsbad,116022.0,115034.0,988.0
616,Chula Vista,281850.0,280211.0,1639.0
617,Coronado,22687.0,17283.0,5404.0
618,Del Mar,3937.0,3937.0,0.0
619,El Cajon,105449.0,102949.0,2500.0
620,Encinitas,62392.0,61835.0,557.0
621,Escondido,151932.0,149374.0,2558.0
622,Imperial Beach,26362.0,25995.0,367.0
623,La Mesa,61863.0,61169.0,694.0
624,Lemon Grove,28445.0,28080.0,365.0


## Census ACS (2020-2024 5-year estimates)

In [81]:
from getpass import getpass

CALIFORNIA_STATE_FIPS = "06"
SAN_DIEGO_COUNTY_FIPS = "073"

CENSUS_API_KEY = getpass("Census API key: ").strip()
if not CENSUS_API_KEY:
    raise ValueError("A Census API key is required.")

ACS_VARS = {
    "B01003_001E": "population_total",
    "B25001_001E": "housing_units_total",
    "B25003_002E": "owner_occupied",
    "B25003_003E": "renter_occupied",
}

acs_url = f"https://api.census.gov/data/{ACS_DATA_YEAR}/acs/acs5"
params = {
    "get": ",".join(["NAME", *ACS_VARS.keys()]),
    "for": "place:*",
    "in": f"state:{CALIFORNIA_STATE_FIPS}",
    "key": CENSUS_API_KEY,
}

acs_raw = get_json(acs_url, params=params)
acs_df = pd.DataFrame(acs_raw[1:], columns=acs_raw[0]).rename(columns=ACS_VARS)
acs_df["jur_clean"] = acs_df["NAME"].str.replace(r" city, California$", "", regex=True).map(normalize_jurisdiction)

sd_acs = acs_df[acs_df["jur_clean"].isin(sd_jur_keys)].copy()
print(f"SD rows ({ACS_VINTAGE_LABEL}):", len(sd_acs))
sd_acs.head()


Census API key:  ········


SD rows (2020-2024): 18


,NAME,population_total,housing_units_total,owner_occupied,renter_occupied,state,place,jur_clean
218,"Carlsbad city, California",114373,47314,27688,16350,06,11194,carlsbad
271,"Chula Vista city, California",276375,90273,51281,34429,06,13392,chula vista
315,"Coronado city, California",19015,9896,3991,3312,06,16378,coronado
364,"Del Mar city, California",3903,2550,987,868,06,18506,del mar
439,"El Cajon city, California",104449,35185,14066,19824,06,21712,el cajon


## Summary

In [83]:
loaded = {
    "APR (permits/completions)": sd_apr_target_year if "sd_apr_target_year" in dir() else pd.DataFrame(),
    "RHNA6 (targets)": sd_rhna6 if "sd_rhna6" in dir() else pd.DataFrame(),
    "City of SD permits": sd_permits_housing if "sd_permits_housing" in dir() else pd.DataFrame(),
    "DOF (population/housing)": sd_dof if "sd_dof" in dir() else pd.DataFrame(),
    "ACS": sd_acs if "sd_acs" in dir() else pd.DataFrame(),
}
for name, df in loaded.items():
    if df.empty:
        print(f"{name:30s} not loaded")
    else:
        n_missing = (df.isna().sum() > 0).sum()
        print(f"{name:30s} {df.shape[0]} rows, {df.shape[1]} cols, {n_missing} cols with missing values")


APR (permits/completions)      7460 rows, 74 cols, 12 cols with missing values
RHNA6 (targets)                19 rows, 16 cols, 0 cols with missing values
City of SD permits             1512 rows, 55 cols, 17 cols with missing values
DOF (population/housing)       19 rows, 17 cols, 0 cols with missing values
ACS                            18 rows, 8 cols, 0 cols with missing values


## Metric dictionary

In [ ]:
metric_dictionary = pd.DataFrame([
    {
        "output_metric": "bp_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of BP_*_INCOME columns",
        "definition": "Total housing units with a building permit issued, all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "co_units_total",
        "source_table": "HCD APR Table A2",
        "source_variables": "Sum of CO_*_INCOME columns",
        "definition": "Total housing units with a certificate of occupancy (completed), all income tiers, per jurisdiction-year",
    },
    {
        "output_metric": "bp_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "bp_affordable_total / bp_units_total",
        "definition": "Share of permitted units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "co_affordable_share",
        "source_table": "HCD APR Table A2",
        "source_variables": "co_affordable_total / co_units_total",
        "definition": "Share of completed units in VLI, LI, or Moderate income categories (excludes Above Moderate)",
    },
    {
        "output_metric": "rhna_target",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "RHNA VLI, RHNA LI, RHNA MOD, RHNA ABOVE MOD",
        "definition": "Assigned RHNA target units per income tier for the 6th Cycle planning period, per jurisdiction",
    },
    {
        "output_metric": "rhna_progress",
        "source_table": "HCD RHNA 6th Cycle Progress Report (Table B)",
        "source_variables": "VLI UNITS, LI UNITS, MOD UNITS, ABOVE MOD UNITS",
        "definition": "Units reported toward the RHNA target so far, per income tier, per jurisdiction. Cumulative cycle-to-date, not year-by-year",
    },
    {
        "output_metric": "sd_permit_du_by_tier",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_DU_EXTREMELY_LOW/VERY_LOW/LOW/MODERATE/ABOVE_MODERATE",
        "definition": "Dwelling units per approval, broken out by income tier, City of San Diego only. Separate ADU (APPROVAL_ADU_*) and JADU (APPROVAL_JADU_*) columns exist alongside standard units",
    },
    {
        "output_metric": "sd_permit_stage",
        "source_table": "City of San Diego Development Permits (Active + Closed approvals)",
        "source_variables": "APPROVAL_ISSUE_DATE, APPROVAL_CLOSE_DATE, approval_status",
        "definition": "This dataset tracks permit issuance and closure only -- it has no certificate-of-occupancy / completion field equivalent to APR's CO_* columns",
    },
])
metric_dictionary.to_csv(DOCS_DIR / "rhna_housing_production_metric_dictionary.csv", index=False)
metric_dictionary


## Export

In [ ]:
output_path = PROCESSED_DIR / f"rhna_housing_production_{TARGET_YEAR}_by_jurisdiction.csv"
production_by_year.to_csv(output_path, index=False)
print("Saved:", output_path)


## Validation against dashboard prototype

In [ ]:
def find_sibling_repo(repo_name: str, search_depth: int = 3) -> Path | None:
    """
    Look for a sibling clone of another repo near this workstream, without
    assuming a fixed folder depth -- works regardless of what each person
    named their local GitHub folder, as long as both repos live under the
    same parent directory somewhere.
    """
    candidates = [ROOT, *ROOT.parents][:search_depth + 2]
    for base in candidates:
        match = base / repo_name
        if match.exists():
            return match
        # also check one level of siblings, in case repos sit in a shared
        # "GitHub" folder rather than directly next to each other
        if base.parent.exists():
            for sibling in base.parent.iterdir():
                if sibling.name == repo_name and sibling.is_dir():
                    return sibling
    return None


baseline_repo = find_sibling_repo("housing-dashboard-prototype")

if baseline_repo is None:
    print(
        "Could not find a local clone of housing-dashboard-prototype near this repo. "
        "Clone it (git clone https://github.com/laurenthanhvo/housing-dashboard-prototype.git) "
        "into the same parent folder as this repo, then re-run this cell."
    )
else:
    BASELINE_PATH = baseline_repo / "data" / "processed" / "sd_apr_a2_city_year_supply.csv"
    print("Found baseline repo at:", baseline_repo)


In [ ]:
if baseline_repo is not None and BASELINE_PATH.exists():
    baseline = pd.read_csv(BASELINE_PATH)
    baseline["jur_clean"] = baseline["jur_clean"].str.lower()

    comparison = production_by_year.merge(
        baseline, on="jur_clean", suffixes=("_fresh", "_baseline"), how="inner",
    )
    failed_checks = comparison[
        comparison["bp_units_total_fresh"] != comparison["bp_units_total_baseline"]
    ]
    print(f"Compared {len(comparison)} rows; {len(failed_checks)} mismatches.")
    display(failed_checks[["jur_clean", "bp_units_total_fresh", "bp_units_total_baseline"]])
    comparison.to_csv(PROCESSED_DIR / "rhna_housing_production_validation_report.csv", index=False)
elif baseline_repo is not None:
    print(f"Repo found but expected file is missing: {BASELINE_PATH}")


## Notes

- Table A2 (`JURIS_NAME`, `YEAR`) is row-level -- one row per project/address,
  not pre-summed by jurisdiction. Production totals require a groupby.
- Each row carries three separate milestone date fields: `ENT_APPROVE_DT1`
  (entitlement), `BP_ISSUE_DT1` (permit), `CO_ISSUE_DT1` (completion), plus
  `NO_ENTITLEMENTS` / `NO_BUILDING_PERMITS` / `NO_OTHER_FORMS_OF_READINESS`
  flags marking which stages don't apply to that row. A single project can
  appear with entitlement, permit, and completion filled in on the same row,
  or split across multiple rows/years for the same APN -- relevant for the
  double-counting check.
- RHNA targets come from Table B (via the separate RHNA progress dataset), not Table A2.
- ACS is pinned to 2024 (2020-2024 vintage) while other sources target 2025;
  any table joining ACS onto 2025 production data mixes two data years.
- `normalize_jurisdiction()` special-cases "National City" -- a blind
  trailing-"city" strip would otherwise collapse it to "national".
